<div dir="rtl" align="right">

# مقایسه و انتخاب مدل رگرسیون

درسی از مدرسه پایتون و هوش مصنوعی، دکتر بهروز طاهری

## کدام مدل را انتخاب کنم؟
هیچ مدلی همیشه بهترین نیست (**قضیه ناهار مجانی وجود ندارد**). راه درست: چند مدل را با **اعتبارسنجی متقاطع یکسان** مقایسه کن.

> [R² میانگین ۵ تای اعتبارسنجی روی داده دیابت. روی این داده کوچک و تقریباً خطی، مدل‌های خطی برنده‌اند.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/reg_compare.png)

*R² میانگین ۵ تای اعتبارسنجی روی داده دیابت. روی این داده کوچک و تقریباً خطی، مدل‌های خطی برنده‌اند.*

جالب است: روی داده دیابت، مدل‌های ساده خطی از جنگل و بوستینگ بهترند. داده کوچک است (۴۴۲ نمونه) و رابطه تقریباً خطی؛ مدل‌های پیچیده فقط نویز بیشتری یاد می‌گیرند.

## جدول راهنما
| مدل | مقیاس‌بندی لازم؟ | غیرخطی؟ | تفسیر | سرعت آموزش | داده بزرگ |
|---|---|---|---|---|---|
| خطی | خیر | خیر | عالی | خیلی سریع | عالی |
| Ridge / Lasso / ENet | **بله** | خیر | عالی | خیلی سریع | عالی |
| چندجمله‌ای | **بله** | بله | متوسط | سریع | ویژگی‌ها زیاد می‌شوند |
| KNN | **بله** | بله | کم | فوری (پیش‌بینی کند) | ضعیف |
| درخت | خیر | بله | عالی | سریع | خوب |
| جنگل تصادفی | خیر | بله | متوسط | متوسط | خوب |
| گرادیان بوستینگ | خیر | بله | متوسط | متوسط | عالی (Hist) |
| SVR | **بله** | بله | کم | کند | ضعیف |

## یک دستور کار عملی
- همیشه با یک **مدل پایه** شروع کن: `DummyRegressor` (میانگین) و رگرسیون خطی.
- اگر رابطه غیرخطی است، جنگل تصادفی را امتحان کن (تنظیم کم).
- برای بهترین دقت روی داده جدولی، گرادیان بوستینگ با توقف زودهنگام.
- اگر تفسیر مهم است، Lasso یا یک درخت کم‌عمق.
- مدل نهایی را **یک بار** روی داده آزمون بسنج.

[[lab:reg_all]]

در آزمایشگاه، همه مدل‌ها را روی داده‌های «پله‌ای» و «با داده پرت» امتحان کن. کدام مدل روی داده پله‌ای بهترین است؟ چرا؟



</div>

In [ ]:
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV
from sklearn.model_selection import KFold, cross_validate
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

X, y = load_diabetes(return_X_y=True)
models = {
    "Dummy (mean)": DummyRegressor(),
    "Linear": LinearRegression(),
    "Ridge": make_pipeline(StandardScaler(), RidgeCV()),
    "Lasso": make_pipeline(StandardScaler(), LassoCV(random_state=0)),
    "KNN": make_pipeline(StandardScaler(), KNeighborsRegressor(15)),
    "Tree": DecisionTreeRegressor(max_depth=3, random_state=0),
    "Forest": RandomForestRegressor(200, min_samples_leaf=5, random_state=0),
    "Boosting": GradientBoostingRegressor(n_estimators=150, learning_rate=0.05, max_depth=2, random_state=0),
    "SVR": make_pipeline(StandardScaler(), SVR(C=100, epsilon=10)),
}
cv = KFold(5, shuffle=True, random_state=0)   # the same folds for every model
rows = []
for name, m in models.items():
    r = cross_validate(m, X, y, cv=cv, scoring=["r2", "neg_mean_absolute_error"])
    rows.append([name, r["test_r2"].mean(), -r["test_neg_mean_absolute_error"].mean(), r["fit_time"].mean() * 1000])
print(pd.DataFrame(rows, columns=["model", "R2", "MAE", "fit ms"]).sort_values("R2", ascending=False).round(3).to_string(index=False))

<div dir="rtl" align="right">



## نمودار پیش‌بینی در برابر واقعی
یک ابزار تشخیصی خوب: اگر نقاط روی خط $y=x$ باشند، مدل خوب است. الگوهای منظم در باقی‌مانده یعنی مدل چیزی را ندیده است.



</div>

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_predict

X, y = load_diabetes(return_X_y=True)
pred = cross_val_predict(LinearRegression(), X, y, cv=5)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(y, pred, s=12); ax[0].plot([25, 346], [25, 346], "r--")
ax[0].set(xlabel="actual", ylabel="predicted", title="Predicted vs actual")
ax[1].scatter(pred, y - pred, s=12); ax[1].axhline(0, color="r", ls="--")
ax[1].set(xlabel="predicted", ylabel="residual", title="Residuals")
plt.tight_layout(); plt.show()